# Пакетный инференс и оценка модели в Google Colab

Этот ноутбук предназначен для запуска в Google Colab после того, как базовая модель и TF-IDF-векторизатор уже обучены и сохранены.

Сценарий ноутбука:

- загрузка сохраненной модели и векторизатора;
- загрузка CSV-файла с новостями для пакетной классификации;
- автоматический поиск текстовой колонки по алиасам;
- расчет предсказанных классов и вероятностей по всем строкам;
- сохранение таблицы предсказаний в `CSV`;
- при наличии истинной метки класса — расчет `Accuracy`, `Precision`, `Recall`, `F1-score` и матрицы ошибок.

Ноутбук оформлен с подробными комментариями на русском языке для последующего описания в ВКР.

## 1. Установка библиотек

Эта ячейка нужна для чистого окружения Google Colab.

In [ ]:
!pip install -q pandas numpy scikit-learn joblib requests

## 2. Импорты и вспомогательные функции

В этой ячейке объявляются функции для загрузки артефактов, пакетного инференса и оценки качества на размеченном CSV.

In [ ]:
from __future__ import annotations

import json
import re
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Iterable

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support


TEXT_COLUMN_CANDIDATES = (
    "text",
    "content",
    "article",
    "body",
    "news",
    "title",
    "description",
    "текст",
    "новость",
    "заголовок",
    "описание",
)

LABEL_COLUMN_CANDIDATES = (
    "label",
    "category",
    "class",
    "topic",
    "target",
    "tag",
    "метка",
    "категория",
    "класс",
    "тема",
)


@dataclass(frozen=True)
class BatchNotebookPaths:
    """Хранит каталоги проекта для пакетного инференса и оценки."""

    root: Path
    model_dir: Path
    vectorizer_dir: Path
    inference_report_dir: Path


def build_project_paths(project_root: str | Path) -> BatchNotebookPaths:
    """Создает каталоги проекта внутри Colab или Google Drive."""
    root = Path(project_root).resolve()
    paths = BatchNotebookPaths(
        root=root,
        model_dir=root / "models" / "classifiers",
        vectorizer_dir=root / "models" / "vectorizers",
        inference_report_dir=root / "reports" / "colab_batch_inference",
    )
    for directory in asdict(paths).values():
        Path(directory).mkdir(parents=True, exist_ok=True)
    return paths


def sanitize_filename(name: str) -> str:
    """Подготавливает безопасное имя файла для сохранения артефактов."""
    cleaned_name = "".join(
        symbol if symbol.isalnum() or symbol in {"-", "_", "."} else "_"
        for symbol in name.strip()
    )
    return cleaned_name or "artifact"


def clean_text_value(text: str) -> str:
    """Выполняет ту же базовую очистку текста, что и при обучении."""
    cleaned_text = text.replace("\u00a0", " ").replace("\t", " ").replace("\n", " ")
    cleaned_text = re.sub(r"\s+", " ", cleaned_text)
    cleaned_text = re.sub(r"\s+([,.;:!?])", r"\1", cleaned_text)
    return cleaned_text.strip().lower()


def clean_label_value(label: str) -> str:
    """Нормализует название класса."""
    return re.sub(r"\s+", " ", label).strip()


def find_required_column(columns: Iterable[str], candidates: tuple[str, ...], logical_name: str) -> str:
    """Ищет нужную колонку по набору алиасов."""
    normalized_mapping = {str(column).strip().casefold(): str(column) for column in columns}
    for candidate in candidates:
        if candidate in normalized_mapping:
            return normalized_mapping[candidate]
    raise ValueError(f"Не найдена обязательная колонка `{logical_name}`.")


def load_model_and_vectorizer(model_path: str | Path, vectorizer_path: str | Path):
    """Загружает сохраненные `joblib`-артефакты и проверяет их типы."""
    model = joblib.load(model_path)
    vectorizer = joblib.load(vectorizer_path)
    if not isinstance(model, LogisticRegression):
        raise TypeError("Ожидается модель типа LogisticRegression.")
    if not isinstance(vectorizer, TfidfVectorizer):
        raise TypeError("Ожидается векторизатор типа TfidfVectorizer.")
    return model, vectorizer


def run_batch_inference(dataframe: pd.DataFrame, model: LogisticRegression, vectorizer: TfidfVectorizer) -> tuple[pd.DataFrame, str]:
    """Выполняет пакетную классификацию новостей по всем непустым строкам CSV."""
    text_column = find_required_column(dataframe.columns, TEXT_COLUMN_CANDIDATES, "text")
    result_dataframe = dataframe.copy()
    result_dataframe["cleaned_text"] = (
        result_dataframe[text_column].astype("string").fillna("").map(clean_text_value)
    )

    valid_mask = result_dataframe["cleaned_text"].ne("")
    if int(valid_mask.sum()) == 0:
        raise ValueError("После очистки в CSV не осталось непустых текстов для классификации.")

    feature_matrix = vectorizer.transform(result_dataframe.loc[valid_mask, "cleaned_text"].tolist())
    probabilities = model.predict_proba(feature_matrix)
    predicted_indices = probabilities.argmax(axis=1)
    predicted_labels = [str(model.classes_[index]) for index in predicted_indices]

    result_dataframe["predicted_label"] = pd.Series(pd.NA, index=result_dataframe.index, dtype="string")
    result_dataframe["predicted_probability"] = np.nan

    valid_indices = result_dataframe.index[valid_mask]
    result_dataframe.loc[valid_indices, "predicted_label"] = predicted_labels
    result_dataframe.loc[valid_indices, "predicted_probability"] = [
        round(float(probabilities[row_index, class_index]), 6)
        for row_index, class_index in enumerate(predicted_indices)
    ]

    for class_position, class_label in enumerate(model.classes_):
        probability_column_name = f"probability_{sanitize_filename(str(class_label))}"
        result_dataframe[probability_column_name] = np.nan
        result_dataframe.loc[valid_indices, probability_column_name] = [
            round(float(value), 6) for value in probabilities[:, class_position]
        ]

    return result_dataframe, text_column


def evaluate_batch_predictions(result_dataframe: pd.DataFrame) -> dict | None:
    """Если в CSV есть истинная метка, рассчитывает метрики качества и матрицу ошибок."""
    try:
        label_column = find_required_column(result_dataframe.columns, LABEL_COLUMN_CANDIDATES, "label")
    except ValueError:
        return None

    true_labels = result_dataframe[label_column].astype("string").fillna("").map(clean_label_value)
    predicted_labels = result_dataframe["predicted_label"].astype("string").fillna("")
    valid_mask = true_labels.ne("") & predicted_labels.ne("")

    if int(valid_mask.sum()) == 0:
        return None

    filtered_true = true_labels.loc[valid_mask]
    filtered_predicted = predicted_labels.loc[valid_mask]
    class_labels = sorted(set(filtered_true.tolist()) | set(filtered_predicted.tolist()))

    accuracy = accuracy_score(filtered_true, filtered_predicted)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        filtered_true,
        filtered_predicted,
        average="macro",
        zero_division=0,
    )
    confusion = confusion_matrix(filtered_true, filtered_predicted, labels=class_labels)

    return {
        "label_column": label_column,
        "evaluated_rows": int(valid_mask.sum()),
        "accuracy": round(float(accuracy), 4),
        "precision_macro": round(float(precision_macro), 4),
        "recall_macro": round(float(recall_macro), 4),
        "f1_macro": round(float(f1_macro), 4),
        "class_labels": class_labels,
        "confusion_matrix": confusion.tolist(),
    }


def save_batch_artifacts(result_dataframe: pd.DataFrame, evaluation_report: dict | None, paths: BatchNotebookPaths, source_name: str) -> dict:
    """Сохраняет CSV предсказаний, JSON-отчет и при необходимости CSV матрицы ошибок."""
    base_name = sanitize_filename(source_name)
    predictions_path = paths.inference_report_dir / f"{base_name}_predictions.csv"
    report_path = paths.inference_report_dir / f"{base_name}_report.json"
    result_dataframe.to_csv(predictions_path, index=False, encoding="utf-8")

    confusion_matrix_path = None
    if evaluation_report is not None:
        confusion_matrix_dataframe = pd.DataFrame(
            evaluation_report["confusion_matrix"],
            index=evaluation_report["class_labels"],
            columns=evaluation_report["class_labels"],
        )
        confusion_matrix_dataframe.index.name = "Истинный класс"
        confusion_matrix_dataframe.columns.name = "Предсказанный класс"
        confusion_matrix_path = paths.inference_report_dir / f"{base_name}_confusion_matrix.csv"
        confusion_matrix_dataframe.to_csv(confusion_matrix_path, encoding="utf-8")

    payload = {
        "generated_at": datetime.now().astimezone().isoformat(timespec="seconds"),
        "source_name": source_name,
        "predictions_path": str(predictions_path),
        "report_path": str(report_path),
        "confusion_matrix_path": str(confusion_matrix_path) if confusion_matrix_path else "",
        "rows": int(len(result_dataframe)),
        "columns": list(result_dataframe.columns),
        "evaluation": evaluation_report,
    }
    report_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

    return {
        "predictions_path": predictions_path,
        "report_path": report_path,
        "confusion_matrix_path": confusion_matrix_path,
    }


## 3. Конфигурация запуска

Ниже нужно указать путь к корню проекта, путь к сохраненной модели, путь к TF-IDF-векторизатору и путь к CSV-файлу для пакетной классификации.

Если требуется, можно предварительно смонтировать Google Drive.

In [ ]:
# При необходимости можно раскомментировать строки ниже.
# from google.colab import drive
# drive.mount('/content/drive')

PROJECT_ROOT = "/content/isnews_colab_project"
MODEL_PATH = "/content/isnews_colab_project/models/classifiers/colab_baseline_model.joblib"
VECTORIZER_PATH = "/content/isnews_colab_project/models/vectorizers/colab_baseline_vectorizer.joblib"
BATCH_DATASET_PATH = ""

paths = build_project_paths(PROJECT_ROOT)
print("Каталог проекта:", paths.root)
print("Каталог пакетных отчетов:", paths.inference_report_dir)

## 4. Загрузка модели, векторизатора и CSV-файла

После выполнения этой ячейки будут загружены артефакты модели и входной CSV для пакетного инференса.

In [ ]:
if not BATCH_DATASET_PATH:
    raise ValueError("Укажите путь BATCH_DATASET_PATH к CSV для пакетной классификации.")

model, vectorizer = load_model_and_vectorizer(MODEL_PATH, VECTORIZER_PATH)
batch_dataframe = pd.read_csv(BATCH_DATASET_PATH)

print("Модель:", MODEL_PATH)
print("Векторизатор:", VECTORIZER_PATH)
print("Файл для пакетной классификации:", BATCH_DATASET_PATH)
print("Размер входного CSV:", batch_dataframe.shape)
batch_dataframe.head()

## 5. Пакетная классификация CSV

Эта ячейка строит таблицу предсказанных классов и вероятностей. Если во входном CSV есть колонка истинного класса, оценка будет рассчитана на следующем шаге.

In [ ]:
result_dataframe, detected_text_column = run_batch_inference(batch_dataframe, model, vectorizer)
print("Найдена текстовая колонка:", detected_text_column)
print("Размер таблицы с предсказаниями:", result_dataframe.shape)
result_dataframe.head()

## 6. Оценка качества на размеченном CSV

Если во входном файле есть колонка с истинным классом, ноутбук рассчитает метрики и матрицу ошибок. Если такой колонки нет, этот шаг будет корректно пропущен.

In [ ]:
evaluation_report = evaluate_batch_predictions(result_dataframe)
if evaluation_report is None:
    print("Во входном CSV не найдена колонка истинного класса. Будет сохранен только пакетный инференс.")
else:
    print(json.dumps(evaluation_report, ensure_ascii=False, indent=2))

## 7. Сохранение результатов

В конце ноутбук сохраняет:

- `CSV` с предсказаниями;
- `JSON`-отчет по пакетному инференсу;
- `CSV` матрицы ошибок, если была доступна истинная метка класса.

In [ ]:
saved_artifacts = save_batch_artifacts(
    result_dataframe=result_dataframe,
    evaluation_report=evaluation_report,
    paths=paths,
    source_name=Path(BATCH_DATASET_PATH).stem,
)

for artifact_name, artifact_path in saved_artifacts.items():
    print(f"{artifact_name}: {artifact_path}")